# 03 - BAF Predictive Model Benchmarks

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ["THESIS_QUICK_RUN"] = "0"
    os.environ["THESIS_SYNTHETIC_FALLBACK"] = "0"
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

PR-AUC là metric xếp hạng chính. Mỗi full run dùng ba seeds. Calibration được fit trên nửa đầu
validation, chọn trên nửa sau validation; decision threshold cũng chỉ chọn trên nửa sau validation.
Reference predictor được chọn bằng mean validation raw PR-AUC, không dùng test.

In [ ]:
from src.experiment import run_repeated_predictive_benchmarks

output_dir = OUTPUT_BASE / "03_baf_model_benchmarks"
result = run_repeated_predictive_benchmarks(
    PROJECT_ROOT / "configs/baf.yaml",
    output_dir=output_dir,
    model_names=("mlp", "tabular_resnet", "tree"),
    quick_run=QUICK_RUN,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
)
summary = result["summary"]
print({
    "data_sources": result["data_sources"],
    "seeds": result["seeds"],
    "reference_model_key": result["reference_model_key"],
    "reference_seed": result["reference_seed"],
    "frozen_manifest": str(result["frozen_manifest_path"]),
})
display(summary.round(4))

## Data and calibration

In [ ]:
display(result["data_summary"])
selected_calibration = result["calibration_comparison"].query("selected").copy()
display(selected_calibration.round(5))
display(result["predictive_bootstrap"].round(5))

## Results

In [ ]:
test_metrics = summary.query("split == 'test'").copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(test_metrics["model"], test_metrics["raw_pr_auc_mean"],
            yerr=test_metrics["raw_pr_auc_std"].fillna(0), color="#4C72B0", capsize=4)
axes[0].set_title("Test raw PR-AUC mean ± SD")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(test_metrics["model"], test_metrics["fbeta_mean"],
            yerr=test_metrics["fbeta_std"].fillna(0), color="#55A868", capsize=4)
axes[1].set_title("Test calibrated-threshold F2 mean ± SD")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(output_dir / "predictive_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
history_rows = []
for seed, model_histories in result["histories"].items():
    for model_name, history in model_histories.items():
        history_frame = pd.DataFrame(history)
        best_index = history_frame["validation_pr_auc"].idxmax()
        history_rows.append({
            "seed": seed, "model": model_name, "epochs_run": len(history_frame),
            "best_epoch": int(history_frame.loc[best_index, "epoch"]),
            "best_validation_pr_auc": history_frame.loc[best_index, "validation_pr_auc"],
        })
display(pd.DataFrame(history_rows).round(5))

## Takeaways

In [ ]:
selected_key = result["reference_model_key"]
validation_best = summary.query("split == 'validation' and model_key == @selected_key").iloc[0]
test_reference = summary.query("split == 'test' and model_key == @selected_key").iloc[0]
display(Markdown(
    f"- Reference model selected on validation: **{validation_best['model']}**.\n"
    f"- Mean validation raw PR-AUC: **{validation_best['raw_pr_auc_mean']:.4f}**.\n"
    f"- Locked test raw PR-AUC: **{test_reference['raw_pr_auc_mean']:.4f} ± {test_reference['raw_pr_auc_std']:.4f}**.\n"
    f"- Frozen reference seed: **{result['reference_seed']}**.\n"
    "- Test metrics report the locked protocol; they are not used for model, calibration, or threshold selection."
))